# 02 用户价值评分与分层

在全量样本上建立评分与标签，再从原始字段重算剔除待核对记录后的 915 人。两套结果同时保留。

In [1]:
from pathlib import Path
import sys
# 兼容从项目根目录或 notebooks 目录打开。
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'run_analysis.py').exists() and (p / 'src/analysis.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image


In [2]:
from src.analysis import load_data, score_users, segment_users, summarize_segments, run_segmentation, CORE, POTENTIAL
raw = load_data()

## 构造特征
登录活跃 R 反向归一化；购买频次 F、累计消费 M 正向归一化。浏览行为 I 为浏览时长与页数得分的等权平均。

基础 RFM = 0.2R + 0.3F + 0.5M；综合得分 = RFM × (1 + 0.2I / 100)。综合得分是排序依据，不是概率。

In [3]:
scored = score_users(raw, weight=0.2)
scored[['User_ID','R_Score','F_Score','M_Score','I_Score','RFM_Score','Final_Score']].head()

,User_ID,R_Score,F_Score,M_Score,I_Score,RFM_Score,Final_Score
0,#1,85.714286,77.777778,49.805607,87.285385,65.378994,76.792255
1,#2,50.000000,77.777778,4.256190,76.638400,35.461428,40.896843
2,#3,3.571429,11.111111,74.769797,25.460637,41.432518,43.542314
3,#4,39.285714,33.333333,87.272355,73.136516,61.493320,70.488135
4,#5,96.428571,55.555556,89.830162,13.646357,80.867462,83.074554


## 生成用户标签
标签采用有优先级的阈值规则，并非对综合得分直接分箱。完整规则在 `src/analysis.py` 的 `segment_users` 中；变更阈值或规则顺序可能影响分层。

In [4]:
users = segment_users(scored)
summarize_segments(users)

,用户数,平均累计消费,平均购买频次,平均浏览得分,平均收入,平均综合分,占比(%)
Segment_Name,,,,,,,
一般用户,443,2368.12,5.17,48.80,84305.72,54.31,44.3
一般发展用户,81,1117.95,6.43,49.84,87107.43,52.67,8.1
重要挽留用户,67,4134.42,2.57,46.53,59698.61,58.59,6.7
高收入消费较高低活跃用户,64,3990.70,4.92,44.09,121985.09,65.40,6.4
低价值用户,50,1036.52,1.68,42.79,94734.76,20.87,5.0
重要发展用户,48,3873.96,1.62,48.91,81230.60,65.41,4.8
一般维持用户,47,1055.40,1.47,45.83,91335.23,33.57,4.7
重要价值用户,47,3960.19,7.47,42.28,73500.94,86.82,4.7
重要保持用户,39,4030.23,7.49,48.00,57995.92,75.26,3.9


In [5]:
print('核心人数', int(users.User_Segment.isin(CORE).sum()))
print('潜力人数', int(users.User_Segment.isin(POTENTIAL).sum()))
users.groupby('Group_Type')[['R_Score','F_Score','M_Score','I_Score','Total_Spending']].mean().round(2)

核心人数 70
潜力人数 89


,R_Score,F_Score,M_Score,I_Score,Total_Spending
Group_Type,,,,,
其他,47.94,49.27,46.07,48.36,2363.32
核心,68.01,85.56,80.16,51.75,4029.59
潜力,31.82,45.32,62.85,54.14,3183.54


## 重新计算，而不是只删行
剔除 85 人后重新计算极值、收入分位和摩擦指标分位数，再执行同一套标签规则。只比较共同保留的 915 人，可以区分删除记录和重新评分带来的影响。

In [6]:
full, recomputed, user_comparison, target_comparison = run_segmentation(raw)
print("全量", len(full), "剔除后重算", len(recomputed))
print("同一批保留用户中标签改变", int(user_comparison.label_changed.sum()))
target_comparison

全量 1000 剔除后重算 915
同一批保留用户中标签改变 43


,指标,人数
0,全量目标人数,159
1,原目标中被标记排除,12
2,保留用户按原评分入选,147
3,保留用户重算后入选,151
4,重新评分后离开,0
5,重新评分后新进入,4


In [7]:
user_comparison.loc[user_comparison.label_changed].head(10)

,User_ID,User_Segment_before,Final_Score_before,User_Segment_after,Final_Score_after,label_changed
24,#25,一般用户,57.541682,隐形活跃者,55.147507,True
44,#47,一般发展用户,44.220748,一般维持用户,41.931611,True
49,#53,一般用户,29.029848,高潜沉睡用户,26.588038,True
65,#71,一般发展用户,50.211101,一般维持用户,47.921070,True
78,#85,一般发展用户,38.092464,一般维持用户,35.888498,True
82,#89,一般用户,30.377705,低价值用户,28.040759,True
112,#123,一般用户,75.862598,重要发展用户,73.614705,True
124,#141,一般用户,74.960186,重要发展用户,72.580841,True
128,#145,一般用户,34.575423,低价值用户,32.378959,True
135,#153,一般用户,85.275197,重要发展用户,82.935482,True


## 分析边界
收入、浏览与活跃度提供画像线索，但不能仅据此证明某类用户会对优惠券产生更高响应。下一步比较排序权重，之后使用情景测算与待执行实验评估策略。